# EXP_010 — Light-Evoked Electrophysiology Analysis

*P. eryngii* mycelium signal recording under blue LED stimulation.

**Protocol:** Pulse stimulation via LED-DRV8 board, PWM intensity sweeps.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats as sp_stats
from scipy import signal as sp_signal
from pathlib import Path
import json

# Dark theme for all plots
plt.rcParams.update({
    'figure.facecolor': '#09090B',
    'axes.facecolor': '#09090B',
    'axes.edgecolor': '#3F3F46',
    'axes.labelcolor': '#A1A1AA',
    'text.color': '#E0E0E0',
    'xtick.color': '#71717A',
    'ytick.color': '#71717A',
    'grid.color': '#27272A',
    'figure.dpi': 120,
})

%matplotlib inline
print('Ready ✓')

## 1 — Load Data

Point this to your experiment directory. Each run produces a folder with `session_*.csv` + `protocol.json`.

In [ ]:
# ── EDIT THIS PATH ──────────────────────────────────────────────
EXP_DIR = Path('../data/experiment_20260310_154233_demo')
# ────────────────────────────────────────────────────────────────

csv_file = list(EXP_DIR.glob('session_*.csv'))[0]
df = pd.read_csv(csv_file)
df.columns = df.columns.str.strip()

with open(EXP_DIR / 'protocol.json') as f:
    protocol = json.load(f)

# Estimate sample rate
fs = 1.0 / np.median(np.diff(df['timestamp_s'].values[:50]))

print(f'Loaded:    {csv_file.name}')
print(f'Samples:   {len(df)}')
print(f'Duration:  {df["timestamp_s"].max():.1f} s')
print(f'Rate:      {fs:.1f} S/s')
print(f'Protocol:  {protocol["name"]}')
df.head()

## 2 — Parse Stimulus Events

In [ ]:
def extract_stim_regions(df):
    regions, current = [], None
    for _, row in df.iterrows():
        ev = str(row.get('stim_event', '')).strip()
        if ev == 'onset':
            current = {'start': row['timestamp_s'], 'channel': int(row['stim_ch']),
                       'pwm': int(row['stim_pwm'])}
        elif ev == 'offset' and current:
            current['end'] = row['timestamp_s']
            regions.append(current)
            current = None
    return regions

stim_regions = extract_stim_regions(df)
print(f'Found {len(stim_regions)} stimulus pulses')
for i, r in enumerate(stim_regions):
    print(f'  Stim {i+1}: {r["start"]:.1f}s → {r["end"]:.1f}s  '
          f'(ch={r["channel"]}, PWM={r["pwm"]}, dur={r["end"]-r["start"]:.1f}s)')

## 3 — Full Session Overview

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['timestamp_s'], df['voltage_uv'], color='#22C55E', linewidth=0.7, alpha=0.9)

for r in stim_regions:
    ax.axvspan(r['start'], r['end'], color='#3B82F6', alpha=0.15)

ax.set_xlabel('Time (s)')
ax.set_ylabel('Voltage (µV)')
ax.set_title(f'Full Session — {protocol["name"]}', fontweight='bold')
ax.grid(True, linewidth=0.5)

trace_p = mpatches.Patch(color='#22C55E', label='Voltage')
stim_p = mpatches.Patch(color='#3B82F6', alpha=0.4, label='LED ON')
ax.legend(handles=[trace_p, stim_p], framealpha=0.3, facecolor='#1E1E1E',
          edgecolor='#3F3F46', labelcolor='#CCC', fontsize=9)
plt.tight_layout()
plt.show()

## 4 — Epoch Analysis

Extract pre / during / post windows for each stimulus and overlay them.

In [ ]:
PRE_S = 1.0   # seconds before onset
POST_S = 1.0  # seconds after offset

epochs = []
for r in stim_regions:
    pre = df[(df['timestamp_s'] >= r['start'] - PRE_S) & (df['timestamp_s'] < r['start'])]
    dur = df[(df['timestamp_s'] >= r['start']) & (df['timestamp_s'] <= r['end'])]
    post = df[(df['timestamp_s'] > r['end']) & (df['timestamp_s'] <= r['end'] + POST_S)]
    epochs.append({'pre': pre, 'during': dur, 'post': post,
                   'onset': r['start'], 'offset': r['end']})

# Overlay all during-stimulus traces, time-locked to onset
fig, ax = plt.subplots(figsize=(10, 4))
stim_dur = 0
all_v = []

for i, ep in enumerate(epochs):
    d = ep['during']
    t_rel = d['timestamp_s'].values - ep['onset']
    v = d['voltage_uv'].values
    ax.plot(t_rel, v, color='#3B82F6', alpha=0.35, linewidth=0.8)
    all_v.append(v)
    stim_dur = max(stim_dur, t_rel[-1] if len(t_rel) > 0 else 0)

# Mean trace
max_len = max(len(v) for v in all_v)
padded = np.full((len(all_v), max_len), np.nan)
for i, v in enumerate(all_v):
    padded[i, :len(v)] = v
mean_v = np.nanmean(padded, axis=0)
t_mean = np.arange(max_len) / fs
ax.plot(t_mean, mean_v, color='#FBBF24', linewidth=2, label='Mean')

ax.axvspan(0, stim_dur, color='#3B82F6', alpha=0.08)
ax.set_xlabel('Time from onset (s)')
ax.set_ylabel('Voltage (µV)')
ax.set_title('Onset-Locked Stimulus Responses', fontweight='bold')
ax.legend(framealpha=0.3, facecolor='#1E1E1E', edgecolor='#3F3F46', labelcolor='#CCC')
ax.grid(True, linewidth=0.5)
ax.axhline(0, color='#3F3F46', linewidth=0.5, linestyle='--')
plt.tight_layout()
plt.show()

## 5 — Phase Comparison (Pre vs During vs Post)

In [ ]:
# Compute per-epoch stats
rows = []
for i, ep in enumerate(epochs):
    for phase in ['pre', 'during', 'post']:
        v = ep[phase]['voltage_uv'].values
        if len(v) == 0:
            continue
        rows.append({
            'stimulus': i + 1,
            'phase': phase,
            'mean_uv': np.mean(v),
            'std_uv': np.std(v),
            'rms_uv': np.sqrt(np.mean(v**2)),
            'peak_to_peak': np.max(v) - np.min(v),
        })

stats_df = pd.DataFrame(rows)
display(stats_df.style.format(precision=2))

In [ ]:
# Box plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric, title in [(ax1, 'rms_uv', 'RMS Voltage (µV)'),
                            (ax2, 'peak_to_peak', 'Peak-to-Peak (µV)')]:
    data = [stats_df[stats_df['phase'] == p][metric].values for p in ['pre', 'during', 'post']]
    colors = ['#71717A', '#3B82F6', '#22C55E']
    bp = ax.boxplot(data, labels=['Pre', 'During', 'Post'], patch_artist=True, widths=0.5)
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.5)
    for item in ['whiskers', 'caps']:
        for line in bp[item]:
            line.set_color('#71717A')
    for line in bp['medians']:
        line.set_color('#E0E0E0')
    ax.set_ylabel(title)
    ax.set_title(title, fontweight='bold')
    ax.grid(True, linewidth=0.5)

plt.tight_layout()
plt.show()

## 6 — Statistical Tests

In [ ]:
pre_rms = stats_df[stats_df['phase'] == 'pre']['rms_uv'].values
dur_rms = stats_df[stats_df['phase'] == 'during']['rms_uv'].values
post_rms = stats_df[stats_df['phase'] == 'post']['rms_uv'].values

n = min(len(pre_rms), len(dur_rms), len(post_rms))

if n >= 2:
    t1, p1 = sp_stats.ttest_rel(pre_rms[:n], dur_rms[:n])
    t2, p2 = sp_stats.ttest_rel(dur_rms[:n], post_rms[:n])
    t3, p3 = sp_stats.ttest_rel(pre_rms[:n], post_rms[:n])
    
    results = pd.DataFrame([
        {'Comparison': 'Pre vs During', 't-stat': t1, 'p-value': p1, 'Significant': '✓' if p1 < 0.05 else '✗'},
        {'Comparison': 'During vs Post', 't-stat': t2, 'p-value': p2, 'Significant': '✓' if p2 < 0.05 else '✗'},
        {'Comparison': 'Pre vs Post', 't-stat': t3, 'p-value': p3, 'Significant': '✓' if p3 < 0.05 else '✗'},
    ])
    display(results.style.format({'t-stat': '{:.3f}', 'p-value': '{:.4f}'}))
else:
    print('⚠ Need ≥ 2 stimuli for paired t-test')

## 7 — Power Spectral Density

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

v = df['voltage_uv'].values
nperseg = min(len(v), 256)
freqs, psd = sp_signal.welch(v, fs=fs, nperseg=nperseg, noverlap=nperseg // 2)

ax.semilogy(freqs, psd, color='#A78BFA', linewidth=1)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD (µV²/Hz)')
ax.set_title('Power Spectral Density', fontweight='bold')
ax.set_xlim(0, fs / 2)
ax.grid(True, linewidth=0.5)
plt.tight_layout()
plt.show()

---

## Notes

- **Demo data**: This was a demo run — voltages are simulated. Real data will show actual mycelium responses.
- **Channel**: LED channel 7 was used (M8 on the board). Verify physical wiring matches.
- **Next**: Run with real ADC-24 connection for meaningful analysis.